# MergeSort (sortare prin interclasare)

Tehnica **Divide et Impera**:
1. **Împarte** vectorul în două jumătăți
2. **Stăpânește**: sortează recursiv fiecare jumătate
3. **Combină**: interclasează cele două jumătăți sortate – O(n)

Recurența:

$$T(n) = 2\,T\!\left(\frac{n}{2}\right) + \Theta(n)$$

Cu Teorema Master: $a = 2$, $b = 2$, $f(n) = n = \Theta\!\left(n^{\log_2 2}\right)$ ⇒ cazul 2 ⇒

$$T(n) = \Theta(n \log n)$$

Proprietăți:
- O(n log n) în **toate** cazurile
- **stabil**
- **nu e in-place**: memorie suplimentară O(n)

## 1. Interclasarea a doi vectori sortați

In [1]:
def merge(a, b):
    result, i, j = [], 0, 0
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:          # <= păstrează stabilitatea
            result.append(a[i]); i += 1
        else:
            result.append(b[j]); j += 1
    result.extend(a[i:])
    result.extend(b[j:])
    return result


print(merge([1, 4, 7, 9], [2, 3, 8, 10, 11]))

[1, 2, 3, 4, 7, 8, 9, 10, 11]


## 2. MergeSort recursiv

In [ ]:
def mergesort(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    return merge(mergesort(arr[:mid]), mergesort(arr[mid:]))


print(mergesort([38, 27, 43, 3, 9, 82, 10]))

## 3. Urmărirea recursivității

In [ ]:
def mergesort_verbose(arr, depth=0):
    indent = "    " * depth
    print(f"{indent}împart {arr}")
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = mergesort_verbose(arr[:mid], depth + 1)
    right = mergesort_verbose(arr[mid:], depth + 1)
    result = merge(left, right)
    print(f"{indent}interclasez {left} + {right} -> {result}")
    return result


mergesort_verbose([38, 27, 43, 3, 9, 82, 10])

## 4. Varianta „ca în C/C++”: pe același vector, cu un vector auxiliar

Sortează `a[lo..hi]` și folosește un singur vector auxiliar alocat o dată.

In [ ]:
def mergesort_inplace(a):
    aux = [None] * len(a)

    def sort(lo, hi):
        if lo >= hi:
            return
        mid = (lo + hi) // 2
        sort(lo, mid)
        sort(mid + 1, hi)
        i, j, k = lo, mid + 1, lo
        while i <= mid and j <= hi:
            if a[i] <= a[j]:
                aux[k] = a[i]; i += 1
            else:
                aux[k] = a[j]; j += 1
            k += 1
        while i <= mid:
            aux[k] = a[i]; i += 1; k += 1
        while j <= hi:
            aux[k] = a[j]; j += 1; k += 1
        a[lo:hi + 1] = aux[lo:hi + 1]

    sort(0, len(a) - 1)
    return a


print(mergesort_inplace([5, 2, 4, 6, 1, 3, 2, 6]))

## 5. MergeSort iterativ (bottom-up)

Fără recursivitate: interclasăm secvențe de lungime 1, apoi 2, 4, 8, ...

In [ ]:
def mergesort_bottom_up(arr):
    arr = list(arr)
    width = 1
    while width < len(arr):
        for lo in range(0, len(arr), 2 * width):
            mid = min(lo + width, len(arr))
            hi = min(lo + 2 * width, len(arr))
            arr[lo:hi] = merge(arr[lo:mid], arr[mid:hi])
        width *= 2
    return arr


print(mergesort_bottom_up([38, 27, 43, 3, 9, 82, 10]))

## 6. Aplicație: numărarea inversiunilor

O **inversiune** este o pereche (i, j) cu i < j și v[i] > v[j]. Brut: O(n²).

Cu mergesort, în O(n log n): la interclasare, când luăm un element din jumătatea dreaptă, el formează inversiuni cu **toate** elementele rămase în jumătatea stângă.

In [ ]:
def count_inversions(arr):
    if len(arr) <= 1:
        return arr, 0
    mid = len(arr) // 2
    a, inv_left = count_inversions(arr[:mid])
    b, inv_right = count_inversions(arr[mid:])
    result, i, j, inv = [], 0, 0, inv_left + inv_right
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            result.append(a[i]); i += 1
        else:
            result.append(b[j]); j += 1
            inv += len(a) - i
    result.extend(a[i:]); result.extend(b[j:])
    return result, inv


arr = [2, 4, 1, 3, 5]
print("inversiuni:", count_inversions(arr)[1])   # (2,1), (4,1), (4,3)

## Testare

In [ ]:
import random

def inversions_brute(arr):
    return sum(arr[i] > arr[j] for i in range(len(arr)) for j in range(i + 1, len(arr)))

for _ in range(500):
    arr = [random.randint(-20, 20) for _ in range(random.randint(0, 40))]
    assert mergesort(arr) == sorted(arr)
    assert mergesort_inplace(list(arr)) == sorted(arr)
    assert mergesort_bottom_up(arr) == sorted(arr)
    assert count_inversions(arr)[1] == inversions_brute(arr)
print("toate testele au trecut")